# APRICOT-M in PyHealth: real-time ICU acuity and therapy prediction

<a href="https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/apricotm_mimic4_acuity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the prediction setting of **APRICOT-M** (Contreras et al., *Real-time prediction of intensive care unit patient acuity and therapy requirements using state-space modelling*, Nature Communications 16, 7315, 2025, [doi:10.1038/s41467-025-62121-1](https://doi.org/10.1038/s41467-025-62121-1); authors' code: [iheallab/apricotM](https://github.com/iheallab/apricotM)) with PyHealth:

- **Task** (`ICUAcuityMIMIC4`): every 4 hours of an ICU stay, the clinical events of the past 4 hours and the static data predict 12 outcomes for the next 4 hours: the acuity state (discharge, stable, unstable, deceased), the transitions unstable→stable and stable→unstable, and the stop and onset of mechanical ventilation (MV), vasopressors (VP) and continuous renal replacement therapy (CRRT). An interval is *unstable* when MV, a vasopressor, CRRT or a massive transfusion is present in it.
- **Model** (`APRICOTM`): the events are (time, variable, value) triplets, embedded without resampling or imputation, contextualized by Mamba (selective state-space) blocks, summarized by top-k pooling, fused with a static embedding, with one head per outcome.
- **Baseline**: logistic regression on per-window statistics (min, max, mean, last value and missingness of each variable, plus static features), the feature set the paper uses for its CatBoost baseline.
- **Metrics**: AUROC and AUPRC per outcome, as in the paper's Table 2.

The PyHealth model is a clean reimplementation from the paper (the authors' code is GPL-3.0 and was only read to confirm details). Its Mamba blocks are pure PyTorch, so it runs on CPU, without the CUDA `mamba_ssm` package.

Part 1 runs end to end on the open **MIMIC-IV Clinical Database Demo** (100 patients, no credentials). It is a **smoke test**: it shows that the pipeline runs, but the test split has only about 20 patients and a handful of positives for most outcomes, so its numbers say nothing about how well the model works. Part 2 runs the same code on the full, credentialed MIMIC-IV.

## Setup

On Colab, the next cell installs PyHealth from GitHub master (`sunlabuiuc/PyHealth`), which has APRICOT-M once this PR is merged. After the next PyPI release, `pip install pyhealth` is enough. Elsewhere the cell does nothing.

In [1]:
import sys

if "google.colab" in sys.modules:
    # APRICOT-M is on PyHealth's master branch; after the next PyPI release,
    # `pip install pyhealth` is enough.
    !pip install -q "git+https://github.com/sunlabuiuc/PyHealth"

In [2]:
import os
import time
import urllib.request

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from pyhealth.datasets import MIMIC4EHRDataset, PatientSplit, get_dataloader
from pyhealth.models import APRICOTM
from pyhealth.tasks import ICUAcuityMIMIC4

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("device:", device)

device: cpu


## Part 1: the open MIMIC-IV demo

Download the demo tables the task needs from PhysioNet (open access, about 10 MB).

In [3]:
DEMO_ROOT = "mimic-iv-demo-2.2"
BASE = "https://physionet.org/files/mimic-iv-demo/2.2/"
TABLES = ["hosp/admissions", "hosp/patients", "hosp/labevents", "hosp/d_labitems",
          "icu/icustays", "icu/chartevents", "icu/outputevents",
          "icu/procedureevents", "icu/inputevents"]
for table in TABLES:
    path = os.path.join(DEMO_ROOT, table + ".csv.gz")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + table + ".csv.gz", path)
print("downloaded", len(TABLES), "tables")

downloaded 9 tables


### Helpers

`run` builds the samples with the processors fitted on the training patients only (`set_task(..., split=PatientSplit(...))`: 70% train, 15% validation, 15% test, by patient), trains APRICOT-M with the authors' settings (width 92, 2 Mamba blocks, top-5 pooling, dropout 0.2, Adam with learning rate 3e-4 and weight decay 1e-5, batch size 128, class weights from the training labels) and keeps the epoch with the best mean validation AUROC. The logistic regression baseline gets the same split and class weighting.

An outcome without both classes in the test split has no AUROC; it is shown as `n/a`.

In [4]:
LABELS = ICUAcuityMIMIC4.LABELS
TABLES_NEEDED = ["labevents", "chartevents", "outputevents",
                 "procedureevents", "inputevents", "admissions"]


def auroc_auprc(y_true, y_prob):
    """Per-outcome AUROC and AUPRC (NaN when a class is missing)."""
    scores = {}
    for k, name in enumerate(LABELS):
        y = y_true[:, k]
        if 0 < y.sum() < len(y):
            scores[name] = (roc_auc_score(y, y_prob[:, k]),
                            average_precision_score(y, y_prob[:, k]), int(y.sum()))
        else:
            scores[name] = (float("nan"), float("nan"), int(y.sum()))
    return scores


@torch.no_grad()
def predict(model, loader):
    model.eval()
    y_true, y_prob = [], []
    for batch in loader:
        out = model(**batch)
        y_true.append(out["y_true"].cpu())
        y_prob.append(out["y_prob"].cpu())
    return torch.cat(y_true).numpy(), torch.cat(y_prob).numpy()


def mean_auroc(scores):
    values = [auc for auc, _, _ in scores.values() if not np.isnan(auc)]
    return float(np.mean(values)) if values else float("nan")


def train_apricotm(train_part, train_dl, val_dl, epochs, patience=5,
                   max_seq_len=512):
    model = APRICOTM(train_part, static_key="static", max_seq_len=max_seq_len,
                     pos_weight="balanced").to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
    best, best_state, waited = -1.0, None, 0
    for epoch in range(epochs):
        model.train()
        start, losses = time.time(), []
        for batch in train_dl:
            optimizer.zero_grad()
            loss = model(**batch)["loss"]
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        val = mean_auroc(auroc_auprc(*predict(model, val_dl)))
        print(f"epoch {epoch + 1}: train loss {np.mean(losses):.3f}, "
              f"val mean AUROC {val:.3f} ({time.time() - start:.0f}s)")
        if val > best:
            best, waited = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model


def window_statistics(part):
    """Min, max, mean, last value and missingness per variable, plus static."""
    vocab = part.input_processors["events"].vocab(0)
    n_vars = len(vocab)
    rows, labels = [], []
    for i in range(len(part)):
        sample = part[i]
        tokens, values = sample["events"][0], sample["events"][1].reshape(-1)
        stats = np.zeros((n_vars, 5))
        stats[:, 4] = 1.0  # missing
        for v in torch.unique(tokens).tolist():
            x = values[tokens == v].numpy()
            stats[v] = [x.min(), x.max(), x.mean(), x[-1], 0.0]
        rows.append(np.concatenate([stats.ravel(), sample["static"].numpy()]))
        labels.append(sample["acuity"].numpy())
    return np.array(rows), np.array(labels)


def logistic_baseline(parts):
    (x_train, y_train), (x_test, y_test) = (window_statistics(parts[0]),
                                            window_statistics(parts[2]))
    scaler = StandardScaler().fit(x_train)
    y_prob = np.zeros_like(y_test)
    for k in range(len(LABELS)):
        if 0 < y_train[:, k].sum() < len(y_train):
            clf = LogisticRegression(max_iter=2000, class_weight="balanced")
            clf.fit(scaler.transform(x_train), y_train[:, k])
            y_prob[:, k] = clf.predict_proba(scaler.transform(x_test))[:, 1]
    return auroc_auprc(y_test, y_prob)


def run(root, cache_dir, epochs, batch_size=128, num_workers=1, max_seq_len=512):
    dataset = MIMIC4EHRDataset(root=root, tables=TABLES_NEEDED, cache_dir=cache_dir,
                               num_workers=num_workers)
    parts = dataset.set_task(ICUAcuityMIMIC4(), num_workers=num_workers,
                             split=PatientSplit((0.7, 0.15, 0.15), seed=0))
    for name, part in zip(("train", "val", "test"), parts):
        stays = len({part[i]["stay_id"] for i in range(len(part))})
        print(f"{name}: {len(part)} windows from {stays} ICU stays")
    train_dl, val_dl, test_dl = (
        get_dataloader(p, batch_size=batch_size, shuffle=(i == 0))
        for i, p in enumerate(parts)
    )
    start = time.time()
    model = train_apricotm(parts[0], train_dl, val_dl, epochs,
                           max_seq_len=max_seq_len)
    print(f"APRICOT-M trained in {time.time() - start:.0f}s")
    return {"APRICOT-M": auroc_auprc(*predict(model, test_dl)),
            "LogReg (window stats)": logistic_baseline(parts)}


def show(results):
    names = list(results)
    print("outcome".ljust(20) + "test +".rjust(7)
          + "".join(f"{n[:22]:>24}" for n in names))
    print(" " * 27 + "".join(f"{'AUROC  AUPRC':>24}" for _ in names))
    for label in LABELS:
        positives = results[names[0]][label][2]
        row = label.ljust(20) + str(positives).rjust(7)
        for n in names:
            auc, ap, _ = results[n][label]
            row += ("n/a".rjust(24) if np.isnan(auc)
                    else f"{auc:>17.2f}{ap:>7.2f}")
        print(row)

### Train and evaluate on the demo

The demo gives about 3,000 four-hour windows from about 130 ICU stays. Ten epochs took about 15 minutes on a laptop CPU (much less on a GPU).

**Memory.** The selective scan keeps a state for every event, so batch size 128 with up to 512 events per sample can need a lot of memory. On a small machine or GPU, pass a smaller `batch_size` to `run` (e.g. 32), or read fewer events per sample with `run(..., max_seq_len=256)`.

In [5]:
demo_results = run(DEMO_ROOT, cache_dir="cache-demo", epochs=10)
show(demo_results)

train: 1715 windows from 91 ICU stays
val: 485 windows from 19 ICU stays
test: 816 windows from 26 ICU stays
epoch 1: train loss 1.247, val mean AUROC 0.640 (124s)
epoch 2: train loss 1.225, val mean AUROC 0.695 (104s)
epoch 3: train loss 1.192, val mean AUROC 0.706 (118s)
epoch 4: train loss 1.158, val mean AUROC 0.716 (95s)
epoch 5: train loss 1.168, val mean AUROC 0.698 (88s)
epoch 6: train loss 1.108, val mean AUROC 0.705 (85s)
epoch 7: train loss 1.090, val mean AUROC 0.719 (104s)
epoch 8: train loss 1.042, val mean AUROC 0.711 (88s)
epoch 9: train loss 1.038, val mean AUROC 0.703 (86s)
epoch 10: train loss 0.991, val mean AUROC 0.726 (81s)
APRICOT-M trained in 973s
outcome              test +               APRICOT-M   LogReg (window stats)
                                       AUROC  AUPRC            AUROC  AUPRC
discharge                25             0.76   0.09             0.70   0.06
stable                  352             0.73   0.66             0.77   0.64
unstable        

## Part 2: full MIMIC-IV (credentialed)

MIMIC-IV requires PhysioNet credentialing ([physionet.org/content/mimiciv](https://physionet.org/content/mimiciv/)). Point `MIMIC4_ROOT` at your copy (the folder with `hosp/` and `icu/`), e.g. on a mounted Google Drive, and use a GPU runtime. Building the event table reads `chartevents` once and caches it in `cache_dir`; MIMIC-IV gives several hundred thousand windows, so expect a long first run. The authors train up to 100 epochs with early stopping. If training runs out of GPU memory, pass a smaller `batch_size` (e.g. `run(..., batch_size=32)`) or `max_seq_len`.

In [6]:
MIMIC4_ROOT = "/content/drive/MyDrive/mimic-iv/2.2"  # change me
EPOCHS = 100

if os.path.isdir(MIMIC4_ROOT):
    full_results = run(MIMIC4_ROOT, cache_dir="cache-mimic4", epochs=EPOCHS,
                       num_workers=4)
    show(full_results)
else:
    print("Set MIMIC4_ROOT to your MIMIC-IV folder to run Part 2.")

Set MIMIC4_ROOT to your MIMIC-IV folder to run Part 2.


## Comparison with the paper

APRICOT-M's reported AUROC for 4-hour predictions (Table 2 of the paper; median of 100 bootstrap samples). *Development* is the validation split of the training cohort (University of Florida Health 2014-2017 and 54 eICU hospitals); *External* is MIMIC-IV plus the other 143 eICU hospitals, never seen in training.

| Outcome | Development | External | Prospective | This notebook |
|---|---|---|---|---|
| Discharge | 0.85 | 0.85 | 0.86 | see above |
| Stable | 0.91 | 0.87 | 0.92 | |
| Unstable | 0.92 | 0.88 | 0.92 | |
| Deceased | 0.94 | 0.95 | 0.99 | |
| Unstable→stable | 0.77 | 0.70 | 0.73 | |
| Stable→unstable | 0.77 | 0.77 | 0.71 | |
| MV onset | 0.80 | 0.81 | 0.73 | |
| VP onset | 0.81 | 0.76 | 0.83 | |
| CRRT onset | 0.94 | 0.86 | 0.93 | |

The paper reports the onset of the three therapies; the task and model also give their stop (`mv_stop`, `vp_stop`, `crrt_stop`), as the authors' code does.

**The demo numbers above are a smoke test, not a reproduction.** The demo test split holds about 20 patients and only a handful of positives for most outcomes (see the `test +` column), so its AUROCs are very noisy, and several outcomes have no positive at all.

What differs from the paper even on full data, so numbers are comparable in trend rather than identical:

- **Data and split**: here the model is trained and tested on MIMIC-IV (a random 70/15/15 split by patient). In the paper MIMIC-IV is part of the *external* cohort: the model never trains on it.
- **Features**: the 48 YAIB labs and vitals (with their plausibility bounds and unit conversions) as events, plus age, sex, height and weight. The paper uses its own list of vitals, labs, medications and assessment scores (Supplementary Table S1), plus race, BMI and comorbidities.
- **Labels**: the paper's text counts any alive ICU discharge as *discharge*; the authors' MIMIC-IV code only counts a discharge home within 24 hours. The task follows the text by default (`ICUAcuityMIMIC4(discharge_home_only=True)` switches). Windows without any event are skipped, while the authors drop the whole stay.
- **Model**: a pure-PyTorch Mamba (same equations, no fused CUDA kernel), padding masked out of the scan and the pooling, no positional encoding (disabled in the authors' code too), and class weights on the positive examples instead of on each label's loss term. No hyperparameter search (the authors run 10 Optuna trials); the defaults are the authors' final settings.
- **Calibration and episodes**: the paper recalibrates probabilities with isotonic regression and also reports episode-level (per-stay) metrics; neither is in this notebook. The samples carry `stay_id` and `window`, so episode-level metrics can be computed from the predictions.

## What to check (for the APRICOT-M authors)

1. The MIMIC-IV therapy items: MV = procedureevents 225792 (invasive ventilation); CRRT = 225802, 225803, 225809; massive transfusion = 229620; vasopressors = inputevents 221906, 221289, 229617, 221662, 222315, 221749 (`pyhealth/tasks/acuity_mimic4.py`).
2. The interval rule: a therapy is present in a 4-hour interval when its start-end period overlaps it; intervals start at ICU admission; the last interval is *deceased* when the patient died in the ICU (death up to 1 hour after the recorded ICU discharge counts) and *discharge* otherwise.
3. Whether the 12-output order (`ICUAcuityMIMIC4.LABELS`) and the model's fusion (pooled events plus static embedding, one two-layer head per outcome) match the published model.